# Random restarts vs. informed initialization

In [11]:
# Imports
import ROOT
import array

from tools import scale_out as so

from emm import data as dat
from emm import fitting
from emm import models

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

ROOT.gErrorIgnoreLevel = ROOT.kWarning

fit_options = [
    ROOT.RooFit.PrintLevel(0),
]

In [12]:
# Load data
x = ROOT.RooRealVar("x", "Diphoton Mass [GeV]", 500, 10_000)
data_tree = dat.get_diphoton_data(tree=True)
diphoton_data = ROOT.RooDataSet("mgg", "mgg", ROOT.RooArgSet(x), ROOT.RooFit.Import(data_tree))
n_events = diphoton_data.numEntries()
data_mean = diphoton_data.mean(x)
print(f"Data mean: {data_mean}")

Data mean: 676.7652379849756


In [13]:
# Set up toy model and reference dataset
toy_model = models.f1(x)
toy_model.pdf.fitTo(diphoton_data)

reference_dataset = toy_model.pdf.generate(ROOT.RooArgSet(x), n_events)

[#1] INFO:NumericIntegration -- RooRealIntegral::init(f_1_Int[x]) using numeric integrator RooRombergIntegrator to calculate Int(x)
[#1] INFO:Minimization -- RooAbsMinimizerFcn::setOptimizeConst: activating const optimization
Minuit2Minimizer: Minimize with max-calls 1000 convergence for edm < 1 strategy 1
Minuit2Minimizer : Valid minimum - status = 1
FVAL  = 31033.2097955985919
Edm   = 0.000159485240499530157
Nfcn  = 27
p1	  = 5.71808	 +/-  0.898473	(limited)
p2	  = -0.778647	 +/-  0.0667606	(limited)
[#1] INFO:Minimization -- RooAbsMinimizerFcn::setOptimizeConst: deactivating const optimization
[#1] INFO:NumericIntegration -- RooRealIntegral::init(f_1_Int[x]) using numeric integrator RooRombergIntegrator to calculate Int(x)
[#1] INFO:NumericIntegration -- RooRealIntegral::init(f_1_Int[x]) using numeric integrator RooRombergIntegrator to calculate Int(x)


Warning in <Minuit2>: MnPosDef Matrix forced pos-def by adding to diagonal 0.00459152
Warning in <Minuit2>: Minuit2Minimizer::Minimize Covar was made pos def


In [14]:
# Set up the exponential mixtures tested
model_primitives = []
for k in range(2, 6):
    model_primitives.append(
        models.ModelPrimitive(
            models.ExponentialMixtureModel,
            k, data_mean=data_mean,
            name=f"ExponentialMixture-{k}"
        )
    )

In [19]:
# Configure the pseudo-dataset study and fit the common toy truth model
n_pseudo_datasets = 100
n_restarts = 50
n_retries = 20
base_seed = 12345

# Test overrides
n_pseudo_datasets = 10

In [ ]:
# Get reference fit results for each model primitive
reference_fit_results = {}
for model_primitive in model_primitives:
    print(f"Fitting reference dataset with {model_primitive.name}...")
    fit_result = fitting.fit_random_restarts(
        x, reference_dataset, model_primitive, base_seed,
        n_restarts, n_retries, save=False
    )
    reference_fit_results[model_primitive.name] = fit_result

In [ ]:
def fit_toy_models(
        x, toy_model,
        seed,
        n_restarts, n_retries,
        reference_fit_results
    ):
    # Fit all models to the toy data and compare their AIC values
    results = { "informed": {}, "random": {} }

    ROOT.RooRandom.randomGenerator().SetSeed(seed)
    toy_data = toy_model.pdf.generate(ROOT.RooArgSet(x), 5036)
    data_mean = toy_data.mean(x)

    for k in range(2, 7):
        mp = models.ModelPrimitive(
            models.ExponentialMixtureModel,
            k, data_mean=data_mean,
        )

        informed_model = mp(x)
        reference_parameters = reference_fit_results[mp.name].final_pars
        informed_model.set_parameters(reference_parameters[k])
        informed_fit_result = fitting.fit_n_retries(
            informed_model, toy_data, n_retries
        )
        if informed_fit_result is not None:
            results["informed"][k] = informed_fit_result["nll"]

        random_fit_result = fitting.fit_random_restarts(
            x, toy_data, mp,
            123,
            n_restarts, n_retries
        )
        if random_fit_result is not None:
            results["random"][k] = random_fit_result["nll"]

    return results

# Fit the toy models in parallel
tasks = []
for i in range(n_pseudo_datasets):
    seed = base_seed + i
    task = so.Task(
        fit_toy_models,
        x, toy_model,
        seed, n_restarts, n_retries,
        reference_fit_results
    )
    tasks.append(task)

results = so.run_tasks(tasks)

In [ ]:
# Plot the matched log-likelihood differences
delta_log_likelihoods = np.array(
    [result["delta_log_likelihood"] for result in paired_results]
)
if len(delta_log_likelihoods) == 0:
    raise RuntimeError("No paired fits converged.")

fig, ax = plt.subplots(figsize=(9, 5))
ax.hist(delta_log_likelihoods, bins=30, histtype="stepfilled", alpha=0.7)
ax.axvline(0, color="black", linestyle="--")
ax.set_xlabel(r"$\Delta \log L = \mathrm{NLL}_{\mathrm{propagated}} - \mathrm{NLL}_{\mathrm{random}}$")
ax.set_ylabel("Pseudo-datasets")
ax.set_title(f"Matched fits: {len(paired_results)}/{n_pseudo_datasets}")
print(f"Median delta log likelihood: {np.median(delta_log_likelihoods):.4g}")
print(f"Fraction where random restarts improve the fit: {(delta_log_likelihoods > 0).mean():.1%}")

# Initial Conditions

In [ ]:
pset = {
    'raw_rate_0': (0.3, 2.5, 50),
    'raw_rate_1': (0.3, 2.5, 50),
}

model = emm.ExponentialMixtureModel(x, 2, data_mean=data.mean(x))

df = emm.scan_parameters(
    model, data, pset,
    constant=False, # Not fixing parameters
    # use_condor=False, n_batches=8, cache_name="2D_grid_restarts", remake_cache=True,
    use_condor=True, n_batches=64, cache_name="2D_grid_restarts",# remake_cache=True,
)

fig, axes = emm.plot_pair_profiles(df, pset, plot_contours=False,)
print(f"Minimum NLL: {df['nll'].min()}")
print(f"Best fit parameters: {df.loc[df['nll'].idxmin()]}")

In [ ]:
# Plot the distribution of the NLL
fig, ax = plt.subplots()
m = df['nll'].min()
n, bins, patches = ax.hist(df['nll'], range=(m, m+0.001), bins=50, density=True)
ax.set_xlabel("NLL")
ax.set_ylabel("Density")

In [ ]:
pset = {
    'raw_rate_0': (0.3, 3, 40),
    'raw_rate_1': (0.3, 3, 40),
    'raw_rate_2': (0.3, 3, 40),
}

model = emm.ExponentialMixtureModel(x, 3, data_mean=data.mean(x))

df = emm.scan_parameters(
    model, data, pset,
    constant=False, # Not fixing parameters
    use_condor=True, n_batches=64, cache_name="3D_grid_restarts",# remake_cache=True,
)

fig, axes = emm.plot_pair_profiles(df, pset, plot_contours=False, worst_case=False)
print(f"Minimum NLL: {df['nll'].min()}")
print(f"Best fit parameters: {df.loc[df['nll'].idxmin()]}")

In [ ]:
# Plot the distribution of the NLL
fig, ax = plt.subplots()
m = df['nll'].min()
n, bins, patches = ax.hist(
    df['nll'],
    # range=(m, m+0.005),
    # range=(31090, 31100),
    bins=50,
    # density=True
)
ax.set_xlabel("NLL")
# ax.set_ylabel("Density")

# Does it matter if we get stuck in a local minimum?

In [ ]:
# Use toys to estimate the NLL uncertainty due to data statistics
best_initial = emm.get_initial(2)
worst_initial = emm.get_initial(2, worst_case=True)

model = emm.ExponentialMixtureModel(x, 2, data_mean=data.mean(x))
defaults = {param.GetName(): param.getVal() for param in model.params}

nll = model.pdf.createNLL(data)

# Get worst case nll
model.set_params(worst_initial)
model.pdf.fitTo(data)
nll_worst = model.pdf.createNLL(data)

# Gest best case nll
model.set_params(best_initial)
model.pdf.fitTo(data)
nll_best = model.pdf.createNLL(data)

# Generate toys and calculate the different in NLL between the best and worst initializations
n_toys = 100
nlls_global = []
nlls_local = []
for i in range(n_toys):
    toy = model.pdf.generate(ROOT.RooArgSet(x), data.numEntries())
    # nll = model.pdf.createNLL(toy)
    i_model = emm.ExponentialMixtureModel(x, 2, data_mean=data.mean(x))
    i_model.set_params(initials)
    nll = i_model.pdf.createNLL(toy)
    fit_result = model.pdf.fitTo(toy)
    nlls.append(nll.getVal())

In [ ]:
# Plot the distribution of the nlls and print some statistics
plt.hist(nlls, bins=10)
plt.xlabel("NLL")
plt.ylabel("Number of toys")
plt.title("NLL distribution from toys")
plt.show()
print(f"Mean NLL: {np.mean(nlls)}")
print(f"Std NLL: {np.std(nlls)}")

In [ ]:
initials = {
    'raw_rate_0': 1.295,
    'raw_rate_1': 0.584,
    'raw_rate_2': 1.579,
}

In [ ]:
model = emm.ExponentialMixtureModel(x, 3, data_mean=data.mean(x))
model.set_params(initials)
emm.fit_and_plot(model, data, x)

In [ ]:
# Add two events in the tail
data_copy = data.Clone()
# vals_to_add = [2500]*50
vals_to_add = np.random.normal(3800, 30, size=1)
for val in vals_to_add:
    x.setVal(val)
    index.setVal(data_copy.numEntries())
    data_copy.add(ROOT.RooArgSet(x, index))

print(f"Data entries (with tail events): {data_copy.numEntries()}")

In [ ]:
# model = emm.ExponentialMixtureModel(x, 3, data_mean=data_copy.mean(x))
models = [emm.ExponentialMixtureModel(x, k, data_mean=data_copy.mean(x)) for k in range(2, 5)]
model_labels = [f"{k} Mixtures" for k in range(2, 5)]
fit_results = [m.pdf.fitTo(data_copy, ROOT.RooFit.Save()) for m in models]
emm.plot_fits(data_copy, x, models, model_labels, fit_results,)
# model.set_params(initials)
# model.set_param('raw_rate_2', 0, constant=True)
# emm.fit_and_plot(model, data_copy, x)

In [ ]:
pset = {
    'raw_rate_0': (0.3, 3, 20),
    'raw_rate_1': (0.3, 3, 20),
}

def model_primitive(x, data):
    model = emm.ExponentialMixtureModel(x, 4, data_mean=data.mean(x), ordered_rates=True)
    return model

df = emm.profile_model(x, model_primitive, data, pset)
fig, axes = emm.plot_pair_profiles(df, pset, plot_contours=False,)

In [ ]:
model_2 = emm.ExponentialMixtureModel(x, 2, data_mean=data.mean(x), use_normalization_construction=True)
emm.fit_and_plot(model_2, data, x, minos=True)

In [ ]:
pset = {
    # 'raw_rate_0': (0.01, 5, 20),
    # 'raw_rate_1': (0.3, 2.5, 20),
    'raw_rate_2': (0.3, 5, 20),
    'raw_rate_3': (0.3, 5, 20)
}

model_2_params = {}
for i in range(model_2.n_components):
    r = model_2.raw_rates[i]
    r_nominal = r.getVal()
    r_up = r.getErrorHi()
    r_down = r.getErrorLo()
    model_2_params[f'raw_rate_{i}'] = (r_nominal, r_nominal-r_down, r_nominal+r_up)

def model_primitive(x, data):
    model = emm.ExponentialMixtureModel(x, 4, data_mean=data.mean(x), use_normalization_construction=True, **model_2_params)
    return model

df = emm.profile_model(x, model_primitive, data, pset)
fig, axes = emm.plot_pair_profiles(df, pset)

In [ ]:
#print the minimum
print("Minimum value of the model:", df['nll'].min())
print(f"Parameters at minimum: \n{df.loc[df['nll'].idxmin()]}")
    

In [ ]:
#print the minimum
print("Minimum value of the model:", df['nll'].min())
print(f"Parameters at minimum: \n{df.loc[df['nll'].idxmin()]}")

In [ ]:

grid = np.linspace(0, 10, 100)

def fit_model(supports):
    model = emm.ExponentialMixtureModel(x, len(supports), data_mean=data.mean(x), use_normalization_construction=True)
    for i, support in enumerate(supports):
        model.raw_rates[i].setVal(support)
        model.raw_rates[i].setConstant(True)
    
    model.pdf.fitTo(data)
    weights = [p.getVal() for p in model.weights]

    nll = model.pdf.createNLL(data)
    return {
        "nll": nll.getVal(),
        "weights": weights,
        "supports": supports
    }

# Randomly search through the parameter space, if a new nll is found 

## Do retries help?

In the Dijet example I allow for a large number of retries (60-100) because these fits seem to fail more often. After retrying they usually succeed, but are these results even useful?

In [ ]:
# Imports
import ROOT
import array

from tools import scale_out as so

from emm import data
from emm import fitting
from emm import models

import matplotlib.pyplot as plt
import numpy as np

In [ ]:
# # Load data
x = ROOT.RooRealVar("x", "Diphoton Mass [GeV]", 500, 10_000)
data_tree = data.get_diphoton_data(tree=True)
diphoton_data = ROOT.RooDataSet("mgg", "mgg", ROOT.RooArgSet(x), ROOT.RooFit.Import(data_tree))
n = diphoton_data.numEntries()
data_mean = diphoton_data.mean(x)
print(f"Data mean: {data_mean}")

In [ ]:
result = fitting.fit_random_restarts_until_converged(
    x, diphoton_data,
    models.ModelPrimitive(
        models.ExponentialMixtureModel,
        3,
        data_mean=data_mean,
        name="ExponentialMixture-3"
    ),
    seed=1234,
    n_retries=20,
    save=False,
    print_level=1,
)

In [ ]:
# Fit random restarts
k = 3
all_results = fitting.fit_random_restarts(
    x, diphoton_data,
    models.ModelPrimitive(
        models.ExponentialMixtureModel,
        k,
        data_mean=data_mean,
        name=f"ExponentialMixture-{k}",
        extended=True
    ),
    seed=1234,
    n_restarts=100,
    n_retries=20,
    save=False,
    return_all_results=True,
    use_multiprocessing=True,
)

In [ ]:
# Print results
min_nll = min([r['nll'] for r in all_results if r is not None])
n_close_to_minimum = sum(1 for r in all_results if r is not None and abs(r['nll'] - min_nll) < 1e-3)
print(f"Minimum NLL: {min_nll}")
print(f"Number of fits close to minimum: {n_close_to_minimum} out of {len(all_results)}")
for i, result in enumerate(all_results):
    if result is None:
        continue
    # print(result)
    print(f"Fit {i}: Delta NLL = {result['nll'] - min_nll}, Status = {result['status']}, n_retries = {result['n_retries']}")

In [ ]:
# Load the data
h_obs, h_fit = data.get_dijet_data()

# Get binning from data
xaxis = h_obs.GetXaxis()
boundaries = [xaxis.GetBinLowEdge(i) for i in range(1, xaxis.GetNbins() + 2)]
binning = ROOT.RooBinning(len(boundaries) - 1, array.array('d', boundaries))

x = ROOT.RooRealVar("x", "Dijet Mass [GeV]", boundaries[0], boundaries[-1])
x.setBinning(binning)

dijet_data = ROOT.RooDataHist("ATLAS_dijet_data", "ATLAS dijet data", ROOT.RooArgList(x), h_obs)
print(f"Data mean: {dijet_data.mean(x)}")

# SW -- Dijet Paper Fit
dh_fit = ROOT.RooDataHist("dh_fit", "dh_fit", ROOT.RooArgList(x), h_fit)
pdf_fit = ROOT.RooHistPdf("pdf_fit", "pdf_fit", ROOT.RooArgSet(x), dh_fit)

fit_options = [
    ROOT.RooFit.IntegrateBins(0.001),
    ROOT.RooFit.PrintLevel(-1),
    # ROOT.RooFit.Offset(True),
    ROOT.RooFit.Strategy(2),
    ROOT.RooFit.Save(),
    # ROOT.RooFit.Range("fit_range")
]

In [ ]:
# result = fitting.fit_random_restarts_until_converged(
#     x, dijet_data,
#     models.ModelPrimitive(
#         models.ExponentialMixtureModel,
#         4,
#         data_mean=dijet_data.mean(x),
#         name="ExponentialMixture-4",
#         extended=True
#     ),
#     seed=1234,
#     n_retries=20,
#     save=False,
#     print_level=1,
#     fit_options=fit_options,
# )

In [ ]:
# Fit random restarts
extended_results = fitting.fit_random_restarts(
    x, dijet_data,
    models.ModelPrimitive(
        models.ExponentialMixtureModel,
        4,
        data_mean=1347,
        name="ExponentialMixture-4",
        extended=True,
        n_data=dijet_data.sumEntries()
    ),
    seed=1234,
    n_restarts=100,
    n_retries=100,
    save=False,
    return_all_results=True,
    fit_options=fit_options,
    use_multiprocessing=True,
)

In [ ]:
# Fit random restarts
softmax_results = fitting.fit_random_restarts(
    x, dijet_data,
    models.ModelPrimitive(
        models.ExponentialMixtureModel,
        4,
        data_mean=1347,
        name="ExponentialMixture-4",
        # extended=True,
        n_data=dijet_data.sumEntries()
    ),
    seed=1234,
    n_restarts=100,
    n_retries=100,
    save=False,
    return_all_results=True,
    fit_options=fit_options,
    use_multiprocessing=True,
)

In [ ]:
# Compare results of extended and softmax fits
print(f"Number of successful extended fits: {sum(1 for r in extended_results if r is not None)}/100")
print(f"Number of successful softmax fits: {sum(1 for r in softmax_results if r is not None)}/100")

extended_min_nll = min([r['nll'] for r in extended_results if r is not None])
softmax_min_nll = min([r['nll'] for r in softmax_results if r is not None])

print(f"Extended fit minimum NLL: {extended_min_nll}")
print(f"Softmax fit minimum NLL: {softmax_min_nll}")

extended_best_fit = min(extended_results, key=lambda r: r['nll'] if r is not None else float('inf'))
softmax_best_fit = min(softmax_results, key=lambda r: r['nll'] if r is not None else float('inf'))
extended_pars = [
    (
        extended_best_fit['final_pars'][f"raw_rate_{i}"],
        extended_best_fit['final_pars'][f"raw_weight_{i}"]) for i in range(4)]
softmax_pars = [
    (
        softmax_best_fit['final_pars'][f"raw_rate_{i}"],
        softmax_best_fit['final_pars'][f"raw_weight_{i}"]) for i in range(4)]
extended_pars.sort(key=lambda x: x[0])
softmax_pars.sort(key=lambda x: x[0])
for i in range(4):
    print(f"Component {i}: Extended fit rate = {extended_pars[i][0]}, weight = {extended_pars[i][1]}")
    # transform the softmax weights
    softmax_weight = np.exp(softmax_pars[i][1])/sum(np.exp([softmax_pars[j][1] for j in range(4)]))
    print(f"Component {i}: Softmax fit rate = {softmax_pars[i][0]}, weight = {softmax_weight}")

In [ ]:
# Plot delta NLL vs. n_retries
delta_nlls = []
n_retries = []
extended_min_nll = min([r['nll'] for r in extended_results if r is not None])
for r in extended_results:
    if r is None:
        continue
    if abs(r['nll'] - extended_min_nll) > 100:
        continue
    delta_nlls.append(r['nll']-extended_min_nll)
    n_retries.append(r['n_retries'])

print(f"{len(delta_nlls)} good fits")
fig, ax = plt.subplots()
ax.set_title("Delta NLL vs. Number of Retries (Extended MLE)")
ax.scatter(n_retries, delta_nlls)
ax.set_xlabel("Number of Retries")
ax.set_ylabel("Delta NLL")

In [ ]:
# Plot delta NLL vs. n_retries
delta_nlls = []
n_retries = []
softmax_min_nll = min([r['nll'] for r in softmax_results if r is not None])
for r in softmax_results:
    if r is None:
        continue
    if abs(r['nll'] - softmax_min_nll) > 100:
        continue
    delta_nlls.append(r['nll']-softmax_min_nll)
    n_retries.append(r['n_retries'])

print(f"{len(delta_nlls)} good fits")
fig, ax = plt.subplots()
ax.set_title("Delta NLL vs. Number of Retries (Softmax MLE)")
ax.scatter(n_retries, delta_nlls)
ax.set_xlabel("Number of Retries")
ax.set_ylabel("Delta NLL")

In [ ]:
# Let's try modifying the rate initialization for the softmax approach
softmax_2_results = fitting.fit_random_restarts(
    x, dijet_data,
    models.ModelPrimitive(
        models.ExponentialMixtureModel,
        4,
        data_mean=1347,
        name="ExponentialMixture-4",
        # extended=True,
        random_rates_2=True,
        n_data=dijet_data.sumEntries()
    ),
    seed=1234,
    n_restarts=100,
    n_retries=100,
    save=False,
    return_all_results=True,
    fit_options=fit_options,
    use_multiprocessing=True,
)

In [ ]:
# Plot delta NLL vs. n_retries
delta_nlls = []
n_retries = []
softmax_2_min_nll = min([r['nll'] for r in softmax_2_results if r is not None])
for r in softmax_2_results:
    if r is None:
        continue
    if abs(r['nll'] - softmax_2_min_nll) > 100:
        continue
    delta_nlls.append(r['nll']-softmax_2_min_nll)
    n_retries.append(r['n_retries'])

print(f"{len(delta_nlls)} good fits")
fig, ax = plt.subplots()
ax.set_title("Delta NLL vs. Number of Retries (Softmax 2 MLE)")
ax.scatter(n_retries, delta_nlls)
ax.set_xlabel("Number of Retries")
ax.set_ylabel("Delta NLL")

# Do we need random restarts for the pseudo-data?

Can we use the same minima for all the pseudo-data fits? How do the results change compared to using random restarts for each pseudo-data fit?

In [ ]:
# Imports
import ROOT
import array

from tools import scale_out as so

from emm import data
from emm import fitting
from emm import models
from emm import bias

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

In [ ]:
# Load the data
h_obs, h_fit = data.get_dijet_data()
n_obs = h_obs.Integral()

# Get binning from data
xaxis = h_obs.GetXaxis()
boundaries = [xaxis.GetBinLowEdge(i) for i in range(1, xaxis.GetNbins() + 2)]
binning = ROOT.RooBinning(len(boundaries) - 1, array.array('d', boundaries))

x = ROOT.RooRealVar("x", "Dijet Mass [GeV]", boundaries[0], boundaries[-1])
x.setBinning(binning)

dijet_data = ROOT.RooDataHist("ATLAS_dijet_data", "ATLAS dijet data", ROOT.RooArgList(x), h_obs)
print(f"Data mean: {dijet_data.mean(x)}")

# SW -- Dijet Paper Fit
dh_fit = ROOT.RooDataHist("dh_fit", "dh_fit", ROOT.RooArgList(x), h_fit)
pdf_fit = ROOT.RooHistPdf("pdf_fit", "pdf_fit", ROOT.RooArgSet(x), dh_fit)

fit_options = [
    ROOT.RooFit.IntegrateBins(0.001),
    ROOT.RooFit.PrintLevel(-1),
    # ROOT.RooFit.Offset(True),
    ROOT.RooFit.Strategy(2),
    ROOT.RooFit.Save(),
    # ROOT.RooFit.Range("fit_range")
]

In [ ]:
# Set up models
toy_models = [
    models.Dijet(x)
]

model_primitives = [
    models.ModelPrimitive(models.Dijet)
]
for k in [3, 4, 5]:
    model_primitives.append(
        models.ModelPrimitive(
            models.ExponentialMixtureModel,
            k,
            data_mean=1347,
            name=f"ExponentialMixture-{k}",
        )
    )

# Fit toy models to data
for toy_model in toy_models:
    print(f"Fitting toy model: {toy_model.name}")
    toy_model.pdf.fitTo(dijet_data, *fit_options)

In [ ]:
# Fit the data with the Exponential Mixture Model with 4 components
model_primitive = models.ModelPrimitive(
    models.ExponentialMixtureModel,
    4,
    data_mean=1347,
    name="ExponentialMixture-4"
)

seed = 12345
n_random_restarts = 200
n_retries = 30

toy_data = toy_models[0].pdf.generateBinned(
    ROOT.RooArgSet(x), n_obs
)
toy_data.SetName("toy_data_0")

toy_fit_result = fitting.fit_random_restarts(
    x, toy_data, model_primitive,
    seed, n_random_restarts, n_retries,
    fit_options=fit_options,
    use_multiprocessing=True,
)

In [ ]:
# Do we need random restarts for every pseudo-experiment
# or can we just find an acceptable solution once and 
# use it for all pseudo-experiments?

n_restarts = 100
n_retries = 30

model_primitive = models.ModelPrimitive(
    models.ExponentialMixtureModel,
    4,
    data_mean=1347,
    name="ExponentialMixture-4"
)

# Simulate several pseudo-experiments and fit with k=4 model
random_results = []
non_random_results = []
for i in range(10):
    print(f"Simulating pseudo-experiment {i+1}")
    toy_data = toy_models[0].pdf.generateBinned(
        ROOT.RooArgSet(x), n_obs
    )
    toy_data.SetName(f"toy_data_{i+1}")

    # Fit without random restarts (use the same seed for all pseudo-experiments)
    model = model_primitive(x)
    model.set_params(toy_fit_result['final_pars'])

    non_random_fit_result = fitting.fit_n_retries(
        model, toy_data, n_retries,
        fit_options=fit_options,
        # print_level=2
    )
    non_random_results.append(non_random_fit_result)

    # Fit with random restarts
    seed = 12345*i+1
    random_fit_results = fitting.fit_random_restarts(
        x, toy_data, model_primitive, seed, n_restarts, n_retries,
        fit_options=fit_options,
        use_multiprocessing=True,
        return_all_results=True,
        # print_level=2
    )
    random_results.append(random_fit_results)


In [ ]:
# Compare the parameters of the best fit across pseudo-experiments
for i, (non_random_result, random_fit_results) in enumerate(zip(non_random_results, random_results)):
    print(f"Pseudo-experiment {i+1}")
    best_random_fit_result = min(random_fit_results, key=lambda r: r['nll'] if r is not None else float('inf'))
    rates_fn = lambda r: sorted([r['final_pars'][f'raw_rate_{i}'] for i in range(4)])
    print(f"   Non-random fit NLL: {non_random_result['nll']}, Rates: {rates_fn(non_random_result)}")
    print(f"   Random fit NLL: {best_random_fit_result['nll']}, Rates: {rates_fn(best_random_fit_result)}")